# Los Angeles Airbnb data cleaning

DSO 576 Module 6 — Kyi Sin Thant

Business question: Among listings accepting stays of 30 nights or fewer with observed prices, how does median listed nightly price vary by neighborhood and room type?

Restart the kernel and run all cells. The scripts preserve the source, retain unknown values, flag unusual records, and verify the full dataset. See README and the PDF for the 14-item checklist and rule/reason/check log.

In [1]:
from pathlib import Path
import json
import pandas as pd
from clean_airbnb import clean
from verify_airbnb import verify
ROOT = Path.cwd()
audit = clean()
print("Rows before / after:", audit["input_rows"], audit["output_rows"])
print("Source SHA-256:", audit["source_sha256"])

Rows before / after: 43932 43932
Source SHA-256: 16007ed61afb75ffe44a69e7296bd50a71d61b2cce7cf885eb9d71d1dbd6f289


## Missing values

Keep unknown license values, prices and host identities. All absent review rates belong to zero-review records; no exposure duration is available to impute a rate. The license count includes four literal N/A markers.

In [2]:
pd.read_csv(ROOT / "outputs/missing_counts.csv")

,column,before,after
0,id,0,0
1,name,0,0
2,host_id,181,181
3,host_profile_id,395,395
4,host_name,384,384
5,neighbourhood_group,0,0
6,neighbourhood,0,0
7,latitude,0,0
8,longitude,0,0
9,room_type,0,0


## Types and suspicious values

IDs are text to preserve exact digits. High prices and long stays remain in the dataset with flags. Repeated host IDs do not imply duplicate listings.

In [3]:
print(json.dumps(audit["failed_conversions"], indent=2))
print(json.dumps(audit["flag_counts"], indent=2))
print("Exact duplicates:", audit["exact_duplicate_rows"])
print("Repeated listing-ID rows:", audit["repeated_listing_id_rows"])

{
  "minimum_nights": 0,
  "number_of_reviews": 0,
  "calculated_host_listings_count": 0,
  "availability_365": 0,
  "number_of_reviews_ltm": 0,
  "price": 0,
  "reviews_per_month": 0,
  "latitude": 0,
  "longitude": 0,
  "last_review": 0
}
{
  "room_type_unrecognized": 0,
  "license_missing": 30740,
  "price_missing": 5570,
  "no_reviews": 11311,
  "review_fields_inconsistent": 0,
  "review_counts_inconsistent": 0,
  "price_high_outlier": 1634,
  "price_under_10": 14,
  "minimum_nights_over_365": 24,
  "short_stay_eligible": 39064,
  "short_stay_price_eligible": 34444,
  "coordinate_review": 0,
  "host_identity_missing": 181
}
Exact duplicates: 0
Repeated listing-ID rows: 0


## Metric denominators and sensitivity

Use median prices within neighborhood and room type. Availability is a calendar measure, not realized occupancy. The <=30-night subset is an analytical choice, not a legal designation.

In [4]:
pd.read_csv(ROOT / "outputs/metrics.csv")

,population,listings,price_n,price_sum,price_mean,price_median,availability_n,availability_mean,review_rate_n,review_rate_mean,reviews_total
0,All listings,43932,38362,14697868.0,383.136124,224.0,43932,223.725849,32621,1.393066,1856294
1,Minimum stay <=30 nights,39064,34444,13816156.0,401.119382,240.0,39064,222.920438,29530,1.493065,1765378
2,Observed price excluding high flags,42298,36728,10428124.0,283.928447,213.0,42298,221.801858,31696,1.406817,1828043


In [5]:
pd.read_csv(ROOT / "outputs/segment_summary.csv").head(15)

,neighbourhood,room_type,listings,price_n,median_price,mean_price,availability_n,mean_available_nights,review_rate_n,mean_reviews_per_month
0,Acton,Entire home/apt,12,12,292.5,477.750000,12,315.583333,9,1.420000
1,Acton,Private room,3,3,234.0,194.333333,3,364.333333,2,0.290000
2,Adams-Normandie,Entire home/apt,12,12,144.0,194.000000,12,221.333333,10,0.708000
3,Adams-Normandie,Private room,54,54,54.0,57.962963,54,262.981481,27,0.679259
4,Agoura Hills,Entire home/apt,33,33,457.0,579.787879,33,256.636364,24,0.828333
5,Agoura Hills,Private room,9,9,76.0,79.111111,9,275.111111,7,0.674286
6,Agua Dulce,Entire home/apt,16,16,233.5,495.000000,16,294.312500,13,1.128462
7,Agua Dulce,Private room,6,6,101.0,99.000000,6,205.333333,5,1.438000
8,Alhambra,Entire home/apt,432,432,280.0,306.835648,432,230.032407,387,2.437132
9,Alhambra,Private room,164,164,97.0,123.914634,164,242.792683,140,1.842857


## Five distinct record checks

Expected outcomes are derived from original source values before comparing actual output. The cleaner preserves all source IDs and every observed price.

In [6]:
pd.read_csv(ROOT / "outputs/record_checks.csv", dtype={"id": "string"})

,id,field,original,expected,reason,actual,passed,outlier_flag_checked
0,2708,license,NaN,NaN,Absent registration is unknown; do not invent ...,NaN,True,NaN
1,9140,price,NaN,NaN,No observed nightly price; omit only from pric...,NaN,True,NaN
2,584724,reviews_per_month,NaN,NaN,Never-reviewed listings have no observed month...,NaN,True,NaN
3,2732,host_profile_id,1.462506e+18,1.462506e+18,Large identifiers retain every source digit th...,1.462506e+18,True,NaN
4,98284,price,2.549000e+03,2.549000e+03,High price may be a luxury rental; retain orig...,2.549000e+03,True,True


## Independent full-data and reproducibility checks

The verifier runs the cleaner in a separate Python process, checks all IDs and observed prices, reconciles totals, and compares nine generated files byte for byte.

In [7]:
verification = verify()
assert verification["passed"]

{
  "passed": true,
  "fresh_process_files_identical": [
    "audit.json",
    "cleaned_listings.csv",
    "cleaned_sample.csv",
    "metrics.csv",
    "missing_counts.csv",
    "original_sample.csv",
    "record_checks.csv",
    "record_checks.json",
    "segment_summary.csv"
  ],
  "full_data_checks": [
    "All listing IDs and order preserved",
    "All three identifier columns preserve every digit",
    "All observed prices and price nulls unchanged",
    "30,740 absent licenses remain null",
    "11,311 zero-review listings retain missing review rates",
    "Review and availability totals unchanged",
    "50 real sample rows exactly match cleaned output",
    "Five distinct original-record checks pass"
  ],
  "source_sha256": "16007ed61afb75ffe44a69e7296bd50a71d61b2cce7cf885eb9d71d1dbd6f289"
}


## Real sample

The 50-row sample deliberately includes difficult cases and room/neighborhood-group coverage. It is not representative for estimating market statistics. Full-data metrics above are calculated from the master dataset.

In [8]:
sample = pd.read_csv(ROOT / "outputs/cleaned_sample.csv", dtype="string")
print("Sample rows:", len(sample))
sample.head(10)

Sample rows: 50


,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,...,no_reviews,review_fields_inconsistent,review_counts_inconsistent,price_high_outlier,price_under_10,minimum_nights_over_365,short_stay_eligible,short_stay_price_eligible,coordinate_review,host_identity_missing
0,2708,"Run Runyon, Beaut Furn Mirror Mini-Suit w/ Fir...",3008,1462506333679450606,⁨Chas.⁩,City of Los Angeles,Hollywood,34.09625,-118.34605,Private room,...,False,False,False,False,False,False,True,True,False,False
1,9140,City Place Lofts,28350,1462507093368580991,Wendell,Other Cities,Long Beach,33.77202,-118.18852,Private room,...,False,False,False,False,False,False,False,False,False,False
2,584724,The Charlie WeHo Luxurious 2BDR/2BATH Apartment,2881219,1462611424198830262,Dylan,City of Los Angeles,Beverly Grove,34.08507,-118.3697,Entire home/apt,...,True,False,False,False,False,False,True,True,False,False
3,2732,Zen Life at the Beach,3041,1462506335238008418,Yogapriestess,Other Cities,Santa Monica,34.0044,-118.48095,Private room,...,False,False,False,False,False,False,True,True,False,False
4,98284,World Cup Luxury HQ in Beverly Hills 6BR Pool ...,518902,1462520558981788997,Dan,City of Los Angeles,Sherman Oaks,34.13258,-118.41202,Entire home/apt,...,False,False,False,True,False,False,True,True,False,False
5,6033,Poolside Serenity Studio,11619,1462506670443088558,Sarah,City of Los Angeles,Woodland Hills,34.16887,-118.64478,Entire home/apt,...,False,False,False,False,False,False,True,True,False,False
6,1127572,Modern apartment in West Hollywood,6186422,1462544354913901661,Michail,City of Los Angeles,Beverly Grove,34.0886,-118.37659,Private room,...,False,False,False,False,True,False,False,False,False,False
7,5108359,“洛杉矶”千里之行(旅行购物冶游)始于“足下”《落脚最佳选择驿站》,26396988,1465365829302298020,图逸,Other Cities,Monterey Park,34.05601,-118.14548,Private room,...,False,False,False,False,True,False,False,False,False,False
8,1079335,One Bedroom + Large Private Patio,5936585,1462543063598972347,Farhad,Other Cities,West Hollywood,34.09688,-118.36523,Entire home/apt,...,False,False,False,False,False,True,False,False,False,False
9,2192814,Private Guest House Near Hollywood!,11182400,1462577533940136751,Josh,City of Los Angeles,Fairfax,34.07815,-118.34905,Entire home/apt,...,False,False,False,False,False,True,False,False,False,False


## Remaining limitations

The original scrape/download date and host_profile_id meaning are unknown. License validity and exemption claims are unverified. Missing prices may bias observed-price comparisons. Calendar unavailability includes blocked nights. The snapshot cannot establish bookings, revenue, seasonality or host entry/exit. Codex assisted in preparing this submission; programmatic verification does not claim personal student review.